# STRATEGY 1: Remove Engineered Features + Use Original 445 Features
Simpler XGBoost without extra noise

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import gc
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("="*70)
print("STRATEGY 1: Original 445 Features Only + Simpler XGBoost")
print("="*70)

# ============== LOAD DATA ==============
print("\n[1/6] Loading data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

# Compress dtypes
def compress_dtypes(df):
    for col in df.columns:
        if df[col].dtype != 'object':
            c_min, c_max = df[col].min(), df[col].max()
            if str(df[col].dtype)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

train_df = compress_dtypes(train_df)
test_df = compress_dtypes(test_df)
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

y_train = train_df['TARGET'].values.copy()
test_ids = test_df['ID'].values.copy()

X_train = train_df.drop(['ID', 'TARGET'], axis=1)
X_test = test_df.drop(['ID'], axis=1)

# Remove constant columns
const_cols = list(set(X_train.columns[X_train.nunique() <= 1]) | set(X_test.columns[X_test.nunique() <= 1]))
X_train = X_train.drop(const_cols, axis=1, errors='ignore')
X_test = X_test.drop(const_cols, axis=1, errors='ignore')

# Align columns
common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

del train_df, test_df
gc.collect()
print(f"✓ Train: {X_train.shape}, Test: {X_test.shape}")

# ============== NO feature engineering - use original features only ==============
print("\n[2/6] Using ORIGINAL FEATURES ONLY (no engineering)...")
X_train_feat = X_train.copy()
X_test_feat = X_test.copy()
print(f"✓ Features: {X_train_feat.shape[1]} (original, no engineering)")

# ============== SETUP CV ==============
print("\n[3/6] Setting up 5-Fold CV...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_indices = list(kf.split(X_train_feat))

test_preds = np.zeros(len(X_test_feat))
xgb_scores = []

# ============== SIMPLER XGBoost (reduce overfitting) ==============
print("\n[4/6] Training SIMPLER XGBoost...")
print("  Parameters: max_depth=3, n_estimators=300, high regularization")

xgb_params = {
    'objective': 'reg:squarederror', 'max_depth': 3, 'learning_rate': 0.05,
    'subsample': 0.7, 'colsample_bytree': 0.7, 'lambda': 8.0, 'alpha': 3.0,
    'random_state': 42, 'n_jobs': 4, 'verbosity': 0
}

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    model = xgb.XGBRegressor(**xgb_params, n_estimators=300)
    model.fit(X_tr, y_tr)
    test_preds += model.predict(X_test_feat) / 5
    r2 = r2_score(y_val, model.predict(X_val))
    xgb_scores.append(r2)
    print(f"  Fold {fold_num} R²: {r2:.6f}")
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

avg_r2 = np.mean(xgb_scores)
print(f"\n✓ XGBoost Avg R²: {avg_r2:.6f}")

# ============== CREATE SUBMISSION ==============
print("\n[5/6] Creating submission...")
final_pred = test_preds
submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})
print(f"✓ Submission shape: {submission.shape}")
print(f"✓ Target - Mean: {submission['TARGET'].mean():.6f}, Std: {submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {submission['TARGET'].min():.6f}, Max: {submission['TARGET'].max():.6f}")

# ============== SAVE SUBMISSION ==============
print("\n[6/6] Saving submission...")
submission.to_csv('submission_strategy1.csv', index=False)
print(f"✓ Saved: submission_strategy1.csv")

# ============== SUMMARY ==============
print("\n" + "="*70)
print("STRATEGY 1 RESULTS")
print("="*70)
print(f"✓ Features: 445 (original only)")
print(f"✓ Model: XGBoost (max_depth=3, high regularization)")
print(f"✓ Validation R²: {avg_r2:.6f}")
print(f"\n→ If this R² is HIGHER than 0.213: Use this for submission!")
print(f"→ File: submission_strategy1.csv")
print("="*70)

# STRATEGY 2: Ridge Regression (Linear Model)
Test if linear relationships work better than complex trees

In [ ]:
from sklearn.linear_model import Ridge

print("\n" + "="*70)
print("STRATEGY 2: Ridge Regression (Linear Baseline)")
print("="*70)

print("\n[1/3] Training Ridge Regression (alpha=15)...")
ridge_scores = []
ridge_preds = np.zeros(len(X_test_feat))

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    X_tr, X_val = X_train_feat.iloc[train_idx], X_train_feat.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    model = Ridge(alpha=15.0)
    model.fit(X_tr, y_tr)
    ridge_preds += model.predict(X_test_feat) / 5
    r2 = r2_score(y_val, model.predict(X_val))
    ridge_scores.append(r2)
    print(f"  Fold {fold_num} R²: {r2:.6f}")
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

ridge_avg_r2 = np.mean(ridge_scores)
print(f"\n✓ Ridge Avg R²: {ridge_avg_r2:.6f}")

# ============== CREATE SUBMISSION ==============
print("\n[2/3] Creating submission...")
ridge_submission = pd.DataFrame({'ID': test_ids, 'TARGET': ridge_preds})
print(f"✓ Submission shape: {ridge_submission.shape}")
print(f"✓ Target - Mean: {ridge_submission['TARGET'].mean():.6f}, Std: {ridge_submission['TARGET'].std():.6f}")
print(f"✓ Target - Min: {ridge_submission['TARGET'].min():.6f}, Max: {ridge_submission['TARGET'].max():.6f}")

# ============== SAVE SUBMISSION ==============
print("\n[3/3] Saving submission...")
ridge_submission.to_csv('submission_strategy2_ridge.csv', index=False)
print(f"✓ Saved: submission_strategy2_ridge.csv")

# ============== SUMMARY ==============
print("\n" + "="*70)
print("STRATEGY 2 RESULTS")
print("="*70)
print(f"✓ Model: Ridge Regression (alpha=15)")
print(f"✓ Validation R²: {ridge_avg_r2:.6f}")
print(f"\n→ If this R² is HIGHER than Strategy 1: Use this for submission!")
print(f"→ File: submission_strategy2_ridge.csv")
print("="*70)

# COMPARISON: Which Strategy to Use?

In [ ]:
print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)

comparison = {
    "Strategy 1 (Simple XGBoost)": avg_r2,
    "Strategy 2 (Ridge Regression)": ridge_avg_r2,
    "Original (Complex XGBoost + Engineering)": 0.212928  # from previous run
}

for name, score in sorted(comparison.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:.<50} R² = {score:.6f}")

best_strategy = max(comparison, key=comparison.get)
best_r2 = comparison[best_strategy]

print("\n" + "="*70)
print(f"BEST STRATEGY: {best_strategy}")
print(f"Validation R²: {best_r2:.6f}")
print("="*70)

if "Strategy 1" in best_strategy:
    print("→ SUBMIT: submission_strategy1.csv")
elif "Strategy 2" in best_strategy:
    print("→ SUBMIT: submission_strategy2_ridge.csv")

print("\n" + "="*70)
print("ANALYSIS:")
print("="*70)
if best_r2 > 0.212928:
    improvement = (best_r2 - 0.212928) / 0.212928 * 100
    print(f"✓ Improvement over original: +{improvement:.2f}%")
    print(f"  This should improve test score (currently -0.03)")
else:
    decline = (best_r2 - 0.212928) / 0.212928 * 100
    print(f"⚠ All strategies similar: {decline:.2f}% difference")
    print(f"  Problem may have fundamentally weak predictive signal")
    print(f"  Consider: original features + engineered might be needed")